In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)

print("✅ Libraries loaded successfully!")

In [ ]:
# Load dataset
df = pd.read_csv('../data/ai4i2020.csv')

print("✅ Dataset loaded successfully!")
print("\nShape of dataset:", df.shape)

print("\nColumns in dataset:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
df.head()

In [ ]:
print("=== Dataset Information ===")
df.info()

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Basic Statistics ===")
df.describe()

In [ ]:
print("=== Failure Distribution ===")
print(df['Machine failure'].value_counts())

failure_rate = df['Machine failure'].mean() * 100
print(f"\nFailure Rate: {failure_rate:.2f}%")

# Plot
plt.figure(figsize=(6,4))

df['Machine failure'].value_counts().plot(kind='bar')

plt.title('Machine Failure Distribution')
plt.xlabel('Failure (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.show()

In [ ]:
sensor_cols = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

fig, axes = plt.subplots(3, 2, figsize=(14,10))
axes = axes.flatten()

for i, col in enumerate(sensor_cols):
    axes[i].plot(df[col].values[:500], linewidth=1)
    axes[i].set_title(f'{col} - First 500 Readings')
    axes[i].set_xlabel('Reading Index')
    axes[i].set_ylabel(col)

# Hide empty subplot
axes[-1].axis('off')

plt.suptitle('IoT Sensor Signals Overview', fontsize=14)
plt.tight_layout()

plt.show()

In [ ]:
# Reset index to maintain proper order
df = df.reset_index(drop=True)

window_size = 10

for col in sensor_cols:
    
    # Rolling mean
    df[f'{col}_roll_mean'] = (
        df[col]
        .rolling(window=window_size)
        .mean()
    )
    
    # Rolling standard deviation
    df[f'{col}_roll_std'] = (
        df[col]
        .rolling(window=window_size)
        .std()
    )
    
    # Rolling variance
    df[f'{col}_roll_var'] = (
        df[col]
        .rolling(window=window_size)
        .var()
    )

# Remove NaN rows created due to rolling window
df_rolled = df.dropna().reset_index(drop=True)

print("✅ Rolling features created!")

print("\nOriginal Shape:", df.shape)
print("New Shape:", df_rolled.shape)

print("\nTotal New Features Added:")
print(len([col for col in df_rolled.columns if 'roll' in col]))

print("\nSample Rolling Features:")
rolling_cols = [col for col in df_rolled.columns if 'roll' in col]
print(rolling_cols[:10])

df_rolled.head()

In [ ]:
print("=== Final Dataset Summary ===")

print(f"Total samples: {df_rolled.shape[0]}")
print(f"Total features: {df_rolled.shape[1]}")
print(f"Failure cases: {df_rolled['Machine failure'].sum()}")

failure_rate = (
    df_rolled['Machine failure'].mean()
) * 100

print(f"Failure Rate: {failure_rate:.2f}%")

# Save feature list
with open('../src/feature_list.txt', 'w') as f:
    for col in df_rolled.columns:
        f.write(col + '\n')

print("\n✅ Feature list saved!")
print("📁 Saved at: src/feature_list.txt")